# SNR-attention experiments — SNR-guided Retinexformer

This notebook runs the planned LOL-v2-syn alpha/seed sweep, trains the selected attention-guided model and a matched control from scratch, then evaluates both on the untouched 100-image LOL-v2-syn test set.

**Kaggle settings:** GPU T4 x2, Internet ON. Keep `STAGE = 'sweep'` for the first saved version. Then attach that output to a new version, change `STAGE` to `'final'`, and run it again. The long cell stops after 10 hours, saves resumable checkpoints every 5,000 iterations, and can be re-run with the prior version attached. Download `snr_attention_state.zip` and `snr_attention_test_results.zip` from the notebook output when finished.

Training uses the same fixed validation split, input crops, model initialization seed, and batches for each alpha/control pair. Alpha selection uses validation PSNR only; the official test set is evaluated only after selection. The control is the original model with no SNR gate.

In [ ]:
import glob, json, os, shutil, subprocess, sys, zipfile
from pathlib import Path

os.chdir('/kaggle/working')
if not Path('btp').exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Crazycraftie/low-light-enhancement-btp.git', 'btp'], check=True)
os.chdir('/kaggle/working/btp')
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/caiyuanhao1998/Retinexformer.git', 'baselines/Retinexformer'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'einops', 'gdown', 'lmdb', 'yacs', 'addict', 'future', 'natsort', 'yapf', 'thop', 'timm', 'lpips', 'scikit-image'], check=True)

import torch
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator before running this notebook.'
print('GPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], '| PyTorch:', torch.__version__)

# A later Kaggle version can attach this notebook's previous output and resume checkpoints.
prior_zips = glob.glob('/kaggle/input/**/snr_attention_state.zip', recursive=True)
for archive in prior_zips:
    with zipfile.ZipFile(archive) as zf:
        zf.extractall('/kaggle/working/btp')
    print('Restored prior run state from', archive)

OPT = 'baselines/Retinexformer/Options/RetinexFormer_LOL_v1.yml'
DATA = Path('data/LOLv2/Synthetic')
STAGE = 'sweep'  # Change to 'final' only after attaching a completed sweep output.
assert STAGE in {'sweep', 'final'}

In [ ]:
if not (DATA / 'Train/Low').is_dir():
    attached = glob.glob('/kaggle/input/**/LOLv2/Synthetic/Train/Low', recursive=True)
    if attached:
        source = Path(attached[0]).parents[2]
        Path('data').mkdir(exist_ok=True)
        (Path('data') / 'LOLv2').symlink_to(source, target_is_directory=True)
        print('Using attached LOL-v2 data:', source)
    else:
        Path('/kaggle/tmp').mkdir(exist_ok=True)
        archive = '/kaggle/tmp/lolv2.zip'
        subprocess.run(['gdown', '-q', '1Ou9EljYZW8o5dbDCf9R34FS8Pd8kEp2U', '-O', archive], check=True)
        subprocess.run(['unzip', '-q', '-o', archive, '-d', 'data'], check=True)

assert len(list((DATA / 'Train/Low').glob('*'))) == 900
assert len(list((DATA / 'Test/Low').glob('*'))) == 100
assert len(list((DATA / 'Test/Normal').glob('*'))) == 100
print('LOL-v2-syn train/test pairs:', len(list((DATA / 'Train/Low').glob('*'))), '/', len(list((DATA / 'Test/Low').glob('*'))))

## 1. Alpha sweep with three matched seeds

The 12 short runs are 3 seeds × (control, alpha 2, alpha 4, alpha 8). Each run starts from scratch and uses L1; therefore the sweep isolates the attention gate rather than stacking it with the earlier SNR-weighted loss. At 10,000 iterations this is a screening experiment, not the final training schedule.

In [ ]:
def save_state_zip():
    path = '/kaggle/working/snr_attention_state.zip'
    with zipfile.ZipFile(path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for p in Path('runs/snr_attention').rglob('*'):
            if p.is_file():
                zf.write(p, p.as_posix())
    print('Saved', path)

if STAGE == 'sweep':
    run_pair(sweep, timeout_seconds=33000)
else:
    print('Sweep stage skipped; restored outputs will be summarized below.')
# Save checkpoints BEFORE summarizing, so an interrupted session still has resumable state.
save_state_zip()

rows = []
for task in sweep:
    metrics_path = Path(task['out']) / 'metrics.json'
    if not metrics_path.exists():
        print('Incomplete run (no metrics.json):', task['out'])
        continue
    metrics = json.loads(metrics_path.read_text())
    rows.append({'alpha': task['alpha'], 'seed': task['seed'], 'best_val_psnr': metrics['best_val_psnr']})
for alpha in (0, 2, 4, 8):
    values = [row['best_val_psnr'] for row in rows if row['alpha'] == alpha]
    if values:
        print(f'alpha={alpha}: mean best validation PSNR={sum(values)/len(values):.4f} dB over {len(values)} seeds')

## 2. Train selected variant and control from scratch

Run this section in a later Kaggle version after attaching the sweep output. The 10-hour timeout is intentional: if a full run does not finish, the latest model/optimizer/scheduler state is retained and the next version resumes it. The same seed, data split, and iterations are used for both models.

In [ ]:
sweep_rows = []
for alpha in (2, 4, 8):
    values = []
    for seed in (42, 43, 44):
        metrics = json.loads(Path(f'runs/snr_attention/sweep/alpha{alpha}_seed{seed}/metrics.json').read_text())
        values.append(metrics['best_val_psnr'])
    sweep_rows.append((sum(values) / len(values), alpha))
BEST_ALPHA = max(sweep_rows, key=lambda row: (row[0], -row[1]))[1]
print('Selected alpha from the three-seed validation mean:', BEST_ALPHA)

def run_final_training():
    final_tasks = [
        {'alpha': 0, 'seed': 42, 'iters': 150000, 'lr': 2e-4, 'val_every': 5000,
         'out': 'runs/snr_attention/final/control'},
        {'alpha': BEST_ALPHA, 'seed': 42, 'iters': 150000, 'lr': 2e-4, 'val_every': 5000,
         'out': f'runs/snr_attention/final/snr_attention_a{BEST_ALPHA}'},
    ]
    run_pair(final_tasks, timeout_seconds=36000)
    with zipfile.ZipFile('/kaggle/working/snr_attention_state.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
        for path in Path('runs/snr_attention').rglob('*'):
            if path.is_file():
                zf.write(path, path.as_posix())
    print('Saved resumable full-training state: /kaggle/working/snr_attention_state.zip')

if STAGE == 'final':
    run_final_training()
else:
    print('Full training skipped; attach this version output and change STAGE to final when ready.')

## 3. Final test evaluation

This uses each run's best validation checkpoint and the held-out LOL-v2-syn test split. Do not select or change alpha based on these test results.

In [ ]:
def evaluate_final():
    control = Path('runs/snr_attention/final/control')
    guided = Path(f'runs/snr_attention/final/snr_attention_a{BEST_ALPHA}')
    if not (control / 'metrics.json').exists() or not (guided / 'metrics.json').exists():
        print('Training is incomplete; resumable state was saved. Attach this version output and rerun STAGE=final.')
        return

    for run_dir, name, alpha in ((control, 'Final-control', None),
                                 (guided, f'Final-SNR-a{BEST_ALPHA}', BEST_ALPHA)):
        weights = run_dir / 'best.pth'
        if not weights.exists():
            weights = run_dir / 'final.pth'
        output = Path('results/snr_attention') / name / 'LOLv2-syn'
        cmd = [sys.executable, 'scripts/retinexformer_infer.py', '--repo', 'baselines/Retinexformer',
               '--opt', OPT, '--weights', str(weights), '--input', str(DATA / 'Test/Low'),
               '--output', str(output), '--method', name, '--device', '0', '--no-speed-log']
        if alpha is not None:
            cmd.extend(['--snr-attention', '--alpha', str(alpha)])
        subprocess.run(cmd, check=True)
        subprocess.run([sys.executable, 'scripts/evaluate.py', '--pred', str(output),
                        '--gt', str(DATA / 'Test/Normal'), '--method', name, '--dataset', 'LOLv2-syn'], check=True)

    with zipfile.ZipFile('/kaggle/working/snr_attention_test_results.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
        for root in (Path('results/snr_attention'), Path('results/per_image')):
            for path in root.rglob('*'):
                if path.is_file() and ('Final' in path.name or 'Final' in str(path)):
                    zf.write(path, path.relative_to(Path('results')).as_posix())
        results = Path('results/results.csv')
        if results.exists():
            zf.write(results, 'results.csv')
    print('Saved test predictions and metric tables: /kaggle/working/snr_attention_test_results.zip')

if STAGE == 'final':
    evaluate_final()
else:
    print('Test evaluation skipped until the final training stage.')